**v2:** adds casual-spelling augmentation and a weakest-classes printout. MuRIL trains on a 40% sample of the casual copies to limit GPU time (about 1.3x the old run).

# Data Sprint 2.0 NLP: Hindi/Hinglish intent (56 classes)
Steps: (1) augmentation + TF-IDF baseline -> submit, (2) MuRIL fine-tune, (3) ensemble. Test is ~64% Roman, train ~33% Roman, so validate on the Roman subset too.

In [ ]:
import glob, re, time, random, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
def find(n): return glob.glob(f"/kaggle/input/**/{n}", recursive=True)[0]
train=pd.read_csv(find("train.csv")); test=pd.read_csv(find("test.csv"))
train["q"]=train.query_text.str.lower(); test["q"]=test.query_text.str.lower()
print(train.shape, test.shape)

## 1. Script handling + augmentation

In [ ]:
import re, random
V={'अ':'a','आ':'aa','इ':'i','ई':'ee','उ':'u','ऊ':'oo','ऋ':'ri','ए':'e','ऐ':'ai','ओ':'o','औ':'au','ऑ':'o','ऍ':'e'}
M={'ा':'aa','ि':'i','ी':'ee','ु':'u','ू':'oo','ृ':'ri','े':'e','ै':'ai','ो':'o','ौ':'au','ॉ':'o','ॅ':'e'}
C={'क':'k','ख':'kh','ग':'g','घ':'gh','ङ':'n','च':'ch','छ':'chh','ज':'j','झ':'jh','ञ':'n','ट':'t','ठ':'th','ड':'d','ढ':'dh','ण':'n',
'त':'t','थ':'th','द':'d','ध':'dh','न':'n','प':'p','फ':'f','ब':'b','भ':'bh','म':'m','य':'y','र':'r','ल':'l','व':'v','श':'sh','ष':'sh',
'स':'s','ह':'h','क़':'q','ख़':'kh','ग़':'g','ज़':'z','ड़':'r','ढ़':'rh','फ़':'f','ळ':'l'}
NUM=str.maketrans('०१२३४५६७८९','0123456789')
def translit(s):
    s=s.translate(NUM); out=[]; i=0; n=len(s)
    while i<n:
        ch=s[i]
        if ch in C:
            base=C[ch]; nxt=s[i+1] if i+1<n else ''
            if nxt in M: out.append(base+M[nxt]); i+=2; continue
            if nxt=='्': out.append(base); i+=2; continue
            # inherent 'a': drop at word end
            end = (i+1>=n) or (not ('\u0900'<=nxt<='\u097F'))
            out.append(base if end else base+'a'); i+=1; continue
        if ch in V: out.append(V[ch])
        elif ch in ('ं','ँ'): out.append('n')
        elif ch=='ः': out.append('h')
        elif ch in ('़','्'): pass
        else: out.append(ch)
        i+=1
    return ''.join(out)
def has_dev(s): return bool(re.search(r'[\u0900-\u097F]',s))
VOW='aeiou'
def noise(s,rng,p=0.25):
    w=s.split(); res=[]
    for t in w:
        r=rng.random()
        if len(t)>3 and r<p*0.4:      # drop a vowel
            idx=[k for k,c in enumerate(t) if c in VOW and 0<k<len(t)-1]
            if idx: k=rng.choice(idx); t=t[:k]+t[k+1:]
        elif len(t)>2 and r<p*0.7:    # double a letter
            k=rng.randrange(len(t)); t=t[:k]+t[k]+t[k:]
        elif len(t)>5 and r<p:        # split word
            k=rng.randrange(2,len(t)-2); res.append(t[:k]); t=t[k:]
        res.append(t)
    return ' '.join(res)
def augment(texts,labels,seed=0):
    rng=random.Random(seed); T=[];L=[]
    for t,l in zip(texts,labels):
        if has_dev(t):
            r=translit(t); T.append(r); L.append(l)
            T.append(noise(r,rng)); L.append(l)
        else:
            T.append(noise(t,rng)); L.append(l)
    return T,L

### Casual-spelling copies
Test Hinglish is more casual than train (short vowels, 'mjhe', fillers like 'bhai'). These copies imitate that style.

In [ ]:
import random, re
FILL=["bhai","yaar","zara","jaldi","please","na"]
SHORT={"mujhe":"mjhe","nahi":"nhi","nahin":"nhi","karo":"kro","kardo":"kardo","hai":"h","batao":"btao","bataao":"btao","kya":"kya","mera":"mera","meri":"meri","aap":"ap","kripya":"plz","dikhao":"dikha","dikhaao":"dikha"}
def casual(s,rng):
    out=[]
    for t in s.split():
        if t in SHORT and rng.random()<0.6: t=SHORT[t]
        else:
            if rng.random()<0.6: t=t.replace("aa","a")
            if rng.random()<0.5: t=t.replace("ee","i")
            if rng.random()<0.5: t=t.replace("oo","u")
            if len(t)>4 and rng.random()<0.15:
                idx=[k for k,c in enumerate(t) if c in "aeiou" and 0<k<len(t)-1]
                if idx: k=rng.choice(idx); t=t[:k]+t[k+1:]
        out.append(t)
    if rng.random()<0.3: out.insert(0 if rng.random()<0.5 else len(out), rng.choice(FILL))
    return " ".join(out)

In [ ]:
dev=train.q.map(has_dev)
Xt,Xv,yt,yv,dt,dv=train_test_split(train.q,train.target,dev,test_size=0.2,stratify=train.target,random_state=0)
Xt,Xv,yt,yv=list(Xt),list(Xv),list(yt),np.array(yv); dv=np.array(dv)
at,al=augment(Xt,yt)
Xt_aug, yt_aug = Xt+at, yt+al
# casual-spelling copies of Roman text (SVM uses all of them, MuRIL a 40% sample to save GPU time)
rngc=random.Random(100)
pool=[(t,l) for t,l in zip(Xt_aug,yt_aug) if not has_dev(t)]
cas=[(casual(t,rngc),l) for t,l in pool]
pick=rngc.sample(range(len(cas)),int(0.4*len(cas)))
Xt_svm=Xt_aug+[x for x,_ in cas]; yt_svm=yt_aug+[l for _,l in cas]
Xt_mu=Xt_aug+[cas[j][0] for j in pick]; yt_mu=yt_aug+[cas[j][1] for j in pick]
print("svm rows",len(Xt_svm),"muril rows",len(Xt_mu))
print(len(Xt), "->", len(Xt_aug))
def report(name,p):
    print(name,"all",round(f1_score(yv,p,average="macro"),4),
          "dev",round(f1_score(yv[dv],p[dv],average="macro"),4),
          "roman",round(f1_score(yv[~dv],p[~dv],average="macro"),4))

## 2. TF-IDF + LinearSVC (fast model, also the <50ms fallback)

In [ ]:
from sklearn.pipeline import make_union
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
def make_feats(): return make_union(
    TfidfVectorizer(analyzer="char_wb",ngram_range=(2,5),sublinear_tf=True,min_df=2),
    TfidfVectorizer(analyzer="word",ngram_range=(1,2),sublinear_tf=True))
f=make_feats(); A=f.fit_transform(Xt_svm); B=f.transform(Xv)
svm=LinearSVC(C=0.5,class_weight="balanced").fit(A,yt_svm)
report("svm", svm.predict(B))
def softmax(z,t=1.0):
    z=z/t; z=z-z.max(1,keepdims=True); e=np.exp(z); return e/e.sum(1,keepdims=True)
svm_val=softmax(svm.decision_function(B),t=0.2)   # temperature is tunable
# weakest classes (read their wrong predictions to find patterns)
from sklearn.metrics import f1_score as _f1
pv=svm.predict(B); per=_f1(yv,pv,average=None,labels=sorted(set(yt)))
for c_,v_ in sorted(zip(sorted(set(yt)),per),key=lambda x:x[1])[:8]: print("class",c_,"F1",round(v_,3))

In [ ]:
# Final TF-IDF model on ALL train (+aug), write first submission
at2,al2=augment(list(train.q),list(train.target),seed=1)
Xall=list(train.q)+at2; yall=list(train.target)+al2
rngc2=random.Random(101)
poolA=[(t,l) for t,l in zip(Xall,yall) if not has_dev(t)]
casA=[(casual(t,rngc2),l) for t,l in poolA]
pickA=rngc2.sample(range(len(casA)),int(0.4*len(casA)))
Xall_svm=Xall+[x for x,_ in casA]; yall_svm=yall+[l for _,l in casA]
Xall_mu=Xall+[casA[j][0] for j in pickA]; yall_mu=yall+[casA[j][1] for j in pickA]
fF=make_feats(); AF=fF.fit_transform(Xall_svm)
svmF=LinearSVC(C=0.5,class_weight="balanced").fit(AF,yall_svm)
svm_test_dec=svmF.decision_function(fF.transform(test.q))
sub=pd.DataFrame({"example_id":test.example_id,"target":svmF.classes_[svm_test_dec.argmax(1)]})
sub.to_csv("submission.csv",index=False); print(sub.target.value_counts().head())

**Submit `submission.csv` now** to get a first leaderboard score, then continue.

## 3. MuRIL fine-tune (needs GPU + the model available; turn Internet on, or add the model as a Kaggle input)
Check the competition rules for allowed pretrained models (<=300M params). `google/muril-base-cased` is about 237M.

In [ ]:
import torch, torch.nn as nn
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
MODEL="google/muril-base-cased"; MAXLEN=48; BS=64; EPOCHS=4; LR=4e-5
dev_=torch.device("cuda")
tok=AutoTokenizer.from_pretrained(MODEL)
def enc(texts): return tok(list(texts),truncation=True,max_length=MAXLEN,padding="max_length",return_tensors="pt")
def train_muril(texts,labels,eval_texts=None,seed=0):
    torch.manual_seed(seed)
    m=AutoModelForSequenceClassification.from_pretrained(MODEL,num_labels=56).to(dev_)
    e=enc(texts); y=torch.tensor(labels)
    n=len(y); steps=EPOCHS*((n+BS-1)//BS)
    opt=torch.optim.AdamW(m.parameters(),lr=LR,weight_decay=0.01)
    sch=get_linear_schedule_with_warmup(opt,int(0.06*steps),steps)
    cnt=np.bincount(labels,minlength=56); w=torch.tensor((cnt.sum()/(56*cnt))**0.5,dtype=torch.float).to(dev_)
    lossf=nn.CrossEntropyLoss(weight=w,label_smoothing=0.05)
    scaler=torch.amp.GradScaler("cuda")
    for ep in range(EPOCHS):
        m.train(); perm=torch.randperm(n)
        for i in range(0,n,BS):
            ix=perm[i:i+BS]
            with torch.autocast("cuda",dtype=torch.float16):
                out=m(input_ids=e["input_ids"][ix].to(dev_),attention_mask=e["attention_mask"][ix].to(dev_)).logits
            loss=lossf(out.float(),y[ix].to(dev_))
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(m.parameters(),1.0); scaler.step(opt); scaler.update(); sch.step()
        print("epoch",ep+1,"done")
    return m
@torch.no_grad()
def predict_proba(m,texts):
    m.eval(); e=enc(texts); P=[]
    for i in range(0,len(texts),256):
        with torch.autocast("cuda",dtype=torch.float16):
            lg=m(input_ids=e["input_ids"][i:i+256].to(dev_),attention_mask=e["attention_mask"][i:i+256].to(dev_)).logits
        P.append(torch.softmax(lg.float(),-1).cpu().numpy())
    return np.concatenate(P)
# validate on the same split
mv=train_muril(Xt_mu,yt_mu)
bert_val=predict_proba(mv,Xv); report("muril",bert_val.argmax(1))

## 4. Ensemble weight on validation, then final fit on all data

In [ ]:
for w in [0.0,0.3,0.5,0.7,1.0]:
    report(f"ens w_bert={w}", (w*bert_val+(1-w)*svm_val).argmax(1))

In [ ]:
W=0.5  # set from the table above
mF=train_muril(Xall_mu,yall_mu)
bert_test=predict_proba(mF,list(test.q))
svm_test=softmax(svm_test_dec,t=0.2)
final=(W*bert_test+(1-W)*svm_test).argmax(1)
sub=pd.DataFrame({"example_id":test.example_id,"target":final}); sub.to_csv("submission.csv",index=False)
print(sub.shape, sub.target.nunique())

## 5. Latency check for the defense (<50 ms)

In [ ]:
q="bhai kal subah 7 baje ka alarm laga de"
t0=time.time()
for _ in range(100): svmF.predict(fF.transform([q.lower()]))
print("tfidf+svm ms/query:",(time.time()-t0)*10)
mF.eval(); t0=time.time()
with torch.no_grad():
    for _ in range(50):
        x=tok([q],return_tensors="pt",truncation=True,max_length=MAXLEN).to(dev_); mF(**x)
torch.cuda.synchronize(); print("muril gpu ms/query:",(time.time()-t0)*20)